In [0]:
%pip install xgboost

In [0]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import joblib
import os

# 1. Path setup
volume_path = '/Volumes/workspace/default/larrydata1/'
train_file = os.path.join(volume_path, 'trainupi.csv')

print("Loading data...")
df = pd.read_csv(train_file)

# 2. --- THE IMPROVED BALANCE ---
# We use 20,000 safe samples to give the model a better sense of 'normal'
fraud_df = df[df['fraud_flag'] == 1]
safe_df = df[df['fraud_flag'] == 0]

safe_sample = safe_df.sample(n=20000, random_state=42)
df_balanced = pd.concat([fraud_df, safe_sample])

print(f"Training on improved set: {len(fraud_df)} Fraud, {len(safe_sample)} Safe.")

# 3. Feature Engineering: Label Encoding
cat_cols = ['transaction type', 'merchant_category', 'sender_age_group', 
            'receiver_age_group', 'sender_state', 'sender_bank', 
            'receiver_bank', 'device_type', 'network_type', 'day_of_week']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    # Using the balanced df to fit encoders
    df_balanced[col] = le.fit_transform(df_balanced[col].astype(str).fillna('Unknown'))
    encoders[col] = le 

# Define Features (X) and Target (y)
X = df_balanced.drop(['transaction id', 'timestamp', 'fraud_flag', 'transaction_status'], axis=1)
y = df_balanced['fraud_flag']

# 4. The Split (Internal to Databricks)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Updated Model Parameters for better Precision/Recall balance
print("Training model (Aggressive learning)...")
model = xgb.XGBClassifier(
    objective='binary:logistic', 
    n_estimators=200,      # Increased for better learning
    learning_rate=0.05,    # Slower learning for better precision
    max_depth=6,           # Slightly deeper to find complex patterns
    scale_pos_weight=20,   # Heavy weight to prioritize the 392 fraud cases
    random_state=42
)
model.fit(X_train, y_train)

# 6. Better Threshold Tuning
print("Optimizing threshold...")
y_probs = model.predict_proba(X_val)[:, 1]
# Checking a finer grain of thresholds from 0.1 to 0.95
thresholds = np.arange(0.1, 0.95, 0.02) 
best_f1 = 0
best_threshold = 0.5

for t in thresholds:
    y_pred = (y_probs >= t).astype(int)
    score = f1_score(y_val, y_pred)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print(f"✅ FINAL Optimal Threshold: {best_threshold:.2f} (F1-Score: {best_f1:.4f})")

# 7. SAVE EVERYTHING TO APP DIRECTORY
app_directory = '/Workspace/Users/rohanoruganti123@gmail.com/larry'
print(f"Saving artifacts to {app_directory}...")

# Save model as JSON
model.save_model(os.path.join(app_directory, 'upi_model.json'))

# Save encoders
joblib.dump(encoders, os.path.join(app_directory, 'encoders.pkl'))

# Save metadata (threshold)
joblib.dump({'threshold': best_threshold}, os.path.join(app_directory, 'metadata.pkl'))

print(f"🚀 Success! All files saved to {app_directory}")
print("\n✅ Files created:")
print(f"  - upi_model.json")
print(f"  - encoders.pkl") 
print(f"  - metadata.pkl")